# Assignment No. 4: Machine Learning Model Application & Evaluation

## 1. Introduction

The objective of this assignment is to apply and compare multiple Machine Learning models on a real-world dataset. The Titanic dataset is selected to predict whether a passenger survived the Titanic disaster.

The dataset contains passenger information such as age, gender, passenger class, fare, and family details. Different classification models will be trained and evaluated using appropriate performance metrics.

## 2. Problem Definition

This is a **Classification** problem because the target variable, `Survived`, contains two classes: 0 (did not survive) and 1 (survived).

The goal is to compare multiple Machine Learning models and identify the best-performing model using suitable evaluation metrics.

In [2]:

from google.colab import files
uploaded = files.upload()


Saving train.csv to train.csv


In [3]:

import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())


Dataset loaded successfully!
Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Column names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [4]:

print("Target column: Survived")
print("\nTarget distribution:")
print(df["Survived"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())


Target column: Survived

Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:

X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]
y = df["Survived"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

display(X.head())


Features (X) shape: (891, 7)
Target (y) shape: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [6]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Training features: (712, 7)
Testing features: (179, 7)
Training target: (712,)
Testing target: (179,)


In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Testing missing values:", np.isnan(X_test_processed).sum())


Processed training shape: (712, 10)
Processed testing shape: (179, 10)
Training missing values: 0
Testing missing values: 0


In [9]:

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    trained_models[name] = model
    print(name, "trained successfully!")


Logistic Regression trained successfully!
KNN trained successfully!
Decision Tree trained successfully!
Random Forest trained successfully!


In [10]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_processed)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0)
    })

    print("\nModel:", name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results)
print("\nModel Comparison Table:")
display(results_df.round(3))



Model: Logistic Regression
Confusion Matrix:
[[98 12]
 [23 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


Model: KNN
Confusion Matrix:
[[98 12]
 [21 48]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179


Model: Decision Tree
Confusion Matrix:
[[97 13]
 [29 40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.88      0.82   

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.804,0.793,0.667,0.724
1,KNN,0.816,0.800,0.696,0.744
2,Decision Tree,0.765,0.755,0.580,0.656
3,Random Forest,0.816,0.800,0.696,0.744


In [11]:

overfit_results = []

for name, model in trained_models.items():
    train_pred = model.predict(X_train_processed)
    test_pred = model.predict(X_test_processed)

    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)

    overfit_results.append({
        "Model": name,
        "Training Accuracy": train_accuracy,
        "Testing Accuracy": test_accuracy,
        "Difference": train_accuracy - test_accuracy
    })

overfit_df = pd.DataFrame(overfit_results)
display(overfit_df.round(3))


,Model,Training Accuracy,Testing Accuracy,Difference
0,Logistic Regression,0.808,0.804,0.003
1,KNN,0.851,0.816,0.035
2,Decision Tree,0.865,0.765,0.100
3,Random Forest,0.983,0.816,0.168


## 5. Model Comparison and Evaluation

Four classification models were applied to the Titanic dataset: Logistic Regression, K-Nearest Neighbors (KNN), Decision Tree, and Random Forest. The dataset was divided into 80% training data and 20% testing data.

The models were evaluated using accuracy, precision, recall, F1-score, and confusion matrices.

KNN and Random Forest achieved the highest test accuracy of 81.6% and an F1-score of 0.744. Logistic Regression achieved an accuracy of 80.4%, while Decision Tree achieved 76.5%.

## 6. Final Conclusion

Based on the evaluation results, KNN and Random Forest performed best, with equal accuracy, precision, recall, and F1-score on the test set.

The F1-score was considered an important metric because it balances precision and recall. Recall was also considered because identifying passengers who actually survived is important for understanding the model's performance on the positive class.

The Decision Tree model showed the weakest test performance among the four models. KNN and Random Forest provided the strongest results in this experiment, but further evaluation would be needed to determine whether either model generalizes better.

Training and testing accuracy will be compared to investigate possible overfitting or underfitting. Overall, this assignment demonstrates how multiple classification models can be trained, evaluated, and compared on a real-world dataset.